In [13]:
import sys
!{sys.executable} -m pip install nltk

In [14]:
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context


import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

#download required NLTK resources(packages)(this only needs to be done once)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

# a small corpus of text documents containing special characters, numbers, and extra spaces
corpus = [
    "Hello!!! Welcome to the world world of GenAI... 123.",
    "Students are LEARNING NLP basics right  now!!",
    " Extra spaces are very annoying in text data. ",
    "Can we remove stop-words and special characters? Yes we can!"
]

print("Original Corpus:")
for i, doc in enumerate(corpus):
    print(f"Doc {i+1}: {doc}")


Original Corpus:
Doc 1: Hello!!! Welcome to the world world of GenAI... 123.
Doc 2: Students are LEARNING NLP basics right  now!!
Doc 3:  Extra spaces are very annoying in text data. 
Doc 4: Can we remove stop-words and special characters? Yes we can!


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jsonu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jsonu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jsonu\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jsonu\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [15]:
cleaned_corpus = []

for doc in corpus:
    # convert eveerything to lowercase  
    doc = doc.lower()
    
     # remove numbers and special characters, using regex to keep only alphabets and spaces
    doc = re.sub(r'[^a-z\s]', '', doc)
    
    # convert multiple spaces to a single space and strip leading/trailing spaces
    doc = re.sub(r'\s+', ' ', doc).strip()
    
    cleaned_corpus.append(doc)

print("Cleaned Corpus:")
for i, doc in enumerate(cleaned_corpus):
    print(f"Doc {i+1}: {doc}")

Cleaned Corpus:
Doc 1: hello welcome to the world world of genai
Doc 2: students are learning nlp basics right now
Doc 3: extra spaces are very annoying in text data
Doc 4: can we remove stopwords and special characters yes we can


In [16]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

final_corpus = []

for doc in cleaned_corpus:
     # tokenization: spliting the document into individual words (tokens)
    tokens = word_tokenize(doc)
    
    # removing stopwords and lemmatization: keeping only the root form of the words
    processed_tokens = []
    for word in tokens:
        if word not in stop_words:
            # convert the word to its root form (lemmatization) 
            root_word = lemmatizer.lemmatize(word,pos='n')  # 'v' for verb lemmatization
            processed_tokens.append(root_word)
            
        # joining words back into a single string for future vectorization (like Bag of Words, TF-IDF, etc.)
    final_corpus.append(" ".join(processed_tokens))
print("After Tokenization, Stopwords & Lemmatization:")
for i, doc in enumerate(final_corpus):
    print(f"Doc {i+1}: {doc}")


After Tokenization, Stopwords & Lemmatization:
Doc 1: hello welcome world world genai
Doc 2: student learning nlp basic right
Doc 3: extra space annoying text data
Doc 4: remove stopwords special character yes


In [17]:
from sklearn.feature_extraction.text import CountVectorizer

# binary=True parameter ensures that the output is a binary matrix (1s and 0s) indicating the presence or absence of words in the documents.
ohe_vectorizer = CountVectorizer(binary=True)
ohe_matrix = ohe_vectorizer.fit_transform(final_corpus).toarray()

print("Vocabulary (Features):", ohe_vectorizer.get_feature_names_out())
print("\nOne-Hot Matrix:\n", ohe_matrix)

Vocabulary (Features): ['annoying' 'basic' 'character' 'data' 'extra' 'genai' 'hello' 'learning'
 'nlp' 'remove' 'right' 'space' 'special' 'stopwords' 'student' 'text'
 'welcome' 'world' 'yes']

One-Hot Matrix:
 [[0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 1 0]
 [0 1 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1]]


In [18]:
# default CountVectorizer creates a Bag of Words representation, where the frequency of each word in the document is counted.
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(final_corpus).toarray()

print("Vocabulary:", bow_vectorizer.get_feature_names_out())
print("\nBag of Words Matrix:\n", bow_matrix)

Vocabulary: ['annoying' 'basic' 'character' 'data' 'extra' 'genai' 'hello' 'learning'
 'nlp' 'remove' 'right' 'space' 'special' 'stopwords' 'student' 'text'
 'welcome' 'world' 'yes']

Bag of Words Matrix:
 [[0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 1 2 0]
 [0 1 0 0 0 0 0 1 1 0 1 0 0 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0 0 1 0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0 0 0 1 0 0 1 1 0 0 0 0 1]]


In [19]:
# ngram_range=(2, 2) parameter specifies that we want to extract bigrams (2-word combinations) from the text.
ngram_vectorizer = CountVectorizer(ngram_range=(2, 2))
ngram_matrix = ngram_vectorizer.fit_transform(final_corpus).toarray()

print("Bigram Vocabulary:", ngram_vectorizer.get_feature_names_out())
print("\nN-gram Matrix:\n", ngram_matrix)

Bigram Vocabulary: ['annoying text' 'basic right' 'character yes' 'extra space'
 'hello welcome' 'learning nlp' 'nlp basic' 'remove stopwords'
 'space annoying' 'special character' 'stopwords special'
 'student learning' 'text data' 'welcome world' 'world genai'
 'world world']

N-gram Matrix:
 [[0 0 0 0 1 0 0 0 0 0 0 0 0 1 1 1]
 [0 1 0 0 0 1 1 0 0 0 0 1 0 0 0 0]
 [1 0 0 1 0 0 0 0 1 0 0 0 1 0 0 0]
 [0 0 1 0 0 0 0 1 0 1 1 0 0 0 0 0]]


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(final_corpus).toarray()

print("TF-IDF Matrix:\n", tfidf_matrix)

TF-IDF Matrix:
 [[0.         0.         0.         0.         0.         0.37796447
  0.37796447 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.37796447 0.75592895
  0.        ]
 [0.         0.4472136  0.         0.         0.         0.
  0.         0.4472136  0.4472136  0.         0.4472136  0.
  0.         0.         0.4472136  0.         0.         0.
  0.        ]
 [0.4472136  0.         0.         0.4472136  0.4472136  0.
  0.         0.         0.         0.         0.         0.4472136
  0.         0.         0.         0.4472136  0.         0.
  0.        ]
 [0.         0.         0.4472136  0.         0.         0.
  0.         0.         0.         0.4472136  0.         0.
  0.4472136  0.4472136  0.         0.         0.         0.
  0.4472136 ]]


In [21]:
import sys
!{sys.executable} -m pip install gensim

In [22]:
from gensim.models import Word2Vec
# word2vec requires a list of tokenized sentences (list of lists) as input, so we need to split each document into words.
tokenized_sentences = [doc.split() for doc in final_corpus]
#training the Word2Vec model with the tokenized sentences, specifying the vector size, window size, and minimum count of words to consider.
#vector_size=5 means each word will be represented by a 5-dimensional vector, window=2 means the context window size is 2 (i.e., considering 2 words before and after the target word), and min_count=1 means that words that appear less than once will be ignored.
w2v_model = Word2Vec(sentences=tokenized_sentences, vector_size=5, window=2, min_count=1)

print("Vocabulary Words:", list(w2v_model.wv.index_to_key))
#check specific word vector representation
word_to_check = 'student'
if word_to_check in w2v_model.wv:
    print(f"\nDense Vector for '{word_to_check}':\n", w2v_model.wv[word_to_check])

Vocabulary Words: ['world', 'yes', 'character', 'special', 'stopwords', 'remove', 'data', 'text', 'annoying', 'space', 'extra', 'right', 'basic', 'nlp', 'learning', 'student', 'genai', 'welcome', 'hello']

Dense Vector for 'student':
 [ 0.0114357   0.14883816 -0.01626565 -0.05276828 -0.17506018]


In [23]:
from sklearn.metrics.pairwise import cosine_similarity

print("--- Document Similarity (Doc 1 vs Doc 2) ---\n")



# 1. Bag of Words Similarity
bow_sim = cosine_similarity([bow_matrix[0]], [bow_matrix[1]])
print(f"Bag of Words Similarity: {bow_sim[0][0]:.4f}")

# 2. TF-IDF Similarity
tfidf_sim = cosine_similarity([tfidf_matrix[0]], [tfidf_matrix[1]])
print(f"TF-IDF Similarity: {tfidf_sim[0][0]:.4f}")

#same document similarity (Doc 1 vs Doc 1) to show that the similarity score is 1.0 for the same document
same_doc_sim = cosine_similarity([tfidf_matrix[0]], [tfidf_matrix[0]])
print(f"\nSame Document (Doc 1 vs Doc 1) Similarity: {same_doc_sim[0][0]:.4f} (1.0 means 100% exact match)")

--- Document Similarity (Doc 1 vs Doc 2) ---

Bag of Words Similarity: 0.0000
TF-IDF Similarity: 0.0000

Same Document (Doc 1 vs Doc 1) Similarity: 1.0000 (1.0 means 100% exact match)


In [24]:
print("--- Word Semantic Similarity (Word2Vec) ---\n")

# take two words from the vocabulary and compute their semantic similarity using Word2Vec model
word1 = 'student'
word2 = 'learning'
# gensim's Word2Vec model allows us to compute the semantic similarity between words based on their vector representations. The similarity score ranges from -1 to 1, where 1 indicates that the words are very similar, 0 indicates no similarity, and -1 indicates that the words are opposites.
if word1 in w2v_model.wv and word2 in w2v_model.wv:
    w2v_sim = w2v_model.wv.similarity(word1, word2)
    print(f"Similarity between '{word1}' and '{word2}': {w2v_sim:.4f}")
    
    # check similarity between two other words in the vocabulary
    word3 = 'text'
    word4 = 'data'
    if word3 in w2v_model.wv and word4 in w2v_model.wv:
        w2v_sim_2 = w2v_model.wv.similarity(word3, word4)
        print(f"Similarity between '{word3}' and '{word4}': {w2v_sim_2:.4f}")
else:
    print("Words inside in model vocabulary .")

--- Word Semantic Similarity (Word2Vec) ---

Similarity between 'student' and 'learning': 0.3094
Similarity between 'text' and 'data': -0.2391
